# Fixing RAG Retrieval When the Words Don't Match

**The problem:** a customer asks *"What is the refund window for international orders?"* The correct answer — 60 days — is in the knowledge base. The bot answers **30 days**. The document exists. Retrieval failed anyway.

This notebook reproduces that failure on a small store-policy corpus, then fixes it step by step: **BM25**, **hybrid search with RRF**, **synonym expansion**, **LLM query rewriting**, and **HyDE**. No API keys needed — the LLM steps run on a small local model.

**Setup:** run the install cell, then run all cells top to bottom. In Colab, `Runtime -> Change runtime type -> T4 GPU` makes the LLM cells fast; on CPU they take a few minutes.

In [1]:
%pip install -q sentence-transformers rank_bm25 transformers accelerate

Note: you may need to restart the kernel to use updated packages.


## The corpus: formal policy language vs. everyday customer language

20 short support documents. The trap is deliberate and realistic: the policy documents are written in formal language (*restitution, consignments dispatched abroad, undelivered*), while customers ask in everyday words (*refund window, international orders, never arrives*). That vocabulary gap is the whole demo.

Each test query has a known gold document — the one document that actually answers it.

In [2]:
DOCS = [
 ("returns-domestic", "Domestic Returns",
  "The refund window for domestic orders is 30 days from delivery. Send the "
  "item back unused and in its original packaging. Once we receive it, the "
  "refund is issued to the original payment method."),
 ("returns-intl", "International Returns",
  "Consignments dispatched abroad may be sent back for restitution within "
  "sixty days of receipt. Enclose the completed form and send the parcel to "
  "our overseas returns centre. Restitution is issued to the original payment "
  "method within ten business days of receipt at our warehouse."),
 ("refund-method", "How Refunds Are Paid",
  "Refunds are issued to the original payment method within 5 to 10 business "
  "days after we receive your return. The refund window for processing begins "
  "once the item arrives at our warehouse."),
 ("warranty", "Warranty",
  "All electronics carry a 12-month warranty against manufacturing defects. "
  "If your device develops a fault, contact support for a repair or replacement."),
 ("faulty-returns", "Faulty or Damaged Items",
  "Merchandise received in a non-conforming state may be remitted for a "
  "replacement or restitution within ninety days of receipt. Photograph the "
  "defect and enclose the images with the consignment."),
 ("lost-parcel", "Lost Parcels",
  "Consignments that remain undelivered twenty-one days after dispatch are "
  "classified as lost in transit. The purchase price is remitted in full and "
  "a substitute consignment can be arranged."),
 ("exchanges", "Exchanges",
  "Footwear and apparel in an unsuitable size may be swapped at no cost to "
  "you. Carriage for the swap is covered by us and the substitute is dispatched "
  "within three business days."),
 ("shipping-times", "Delivery Times",
  "Domestic delivery takes 2 to 4 business days. Cross-border delivery takes "
  "7 to 14 business days depending on the destination."),
 ("order-tracking", "Tracking Your Order",
  "Track your parcel from your account page. Tracking updates appear within "
  "24 hours of dispatch."),
 ("payment-methods", "Payment Methods",
  "We accept major credit cards, debit cards and digital wallets. Payment is "
  "taken when your order is dispatched."),
 ("return-shipping-cost", "Return Shipping Costs",
  "Returning a domestic order is free. For cross-border returns, the customer "
  "covers the postage unless the item is faulty."),
 ("gift-returns", "Gift Returns",
  "Gifts can be returned within 30 days for an exchange or store credit. "
  "The buyer is not notified."),
 ("sale-items", "Sale Items",
  "Items bought on sale can be returned within 14 days for an exchange or "
  "store credit. Sale items cannot be refunded to the original payment method."),
 ("store-credit", "Store Credit",
  "Store credit never expires and can be used across all departments, online "
  "and in store."),
 ("damaged-box", "Damaged Packaging",
  "If the outer box arrives damaged but the item is fine, keep the item and "
  "report the packaging so we can claim from the carrier."),
 ("vat-customs", "Customs and Duties",
  "Cross-border orders may attract import duties. Duties are calculated at "
  "checkout and are not refundable once the parcel has shipped."),
 ("size-guide", "Size Guide",
  "Footwear sizes run true to size. If you are between sizes, we recommend "
  "ordering the larger size."),
 ("account", "Your Account",
  "Update your address, view orders and manage payment methods from your "
  "account settings."),
 ("click-collect", "Click and Collect",
  "Order online and collect from any store within 2 hours. Bring your order "
  "confirmation and photo ID."),
 ("price-match", "Price Match",
  "If you find the same item cheaper elsewhere within 14 days, we will match "
  "the price and refund the difference."),
]

QUERIES = [
 ("What is the refund window for international orders?", "returns-intl"),
 ("How long do I have to send back a faulty headset?", "faulty-returns"),
 ("Can I get my money back if the parcel never arrives?", "lost-parcel"),
 ("Do you charge for returning shoes that don't fit?", "exchanges"),
]

In [3]:
import re
import numpy as np

def show(query, ranking, top_k=3):
    """Print the top-k documents of a ranking, so you can see what a method returned.

    query: the question that was asked.
    ranking: a list of positions into DOCS, best first (0 = top hit).
    top_k: how many results to print (default 3).
    Prints one line per result: rank number, title, and id. Returns nothing.
    """
    print(f"Query: {query}")
    for rank, idx in enumerate(ranking[:top_k], 1):
        doc_id, title, _ = DOCS[idx]
        print(f"  {rank}. {title} (id={doc_id})")
    print()

def gold_rank(ranking, gold_id):
    """Return the 1-based rank of the gold (correct) document in a ranking.

    ranking: a list of positions into DOCS, best first.
    gold_id: the id of the document that actually answers the query.
    Returns 1 if the gold doc is the top hit, 2 if second, and so on.
    This is the number the whole scoreboard is built from.
    """
    ids = [d[0] for d in DOCS]
    return ranking.index(ids.index(gold_id)) + 1

## Step 1: Baseline — pure vector search

Embed every document with `all-MiniLM-L6-v2`, embed the query, rank by cosine similarity. This is the default RAG setup most people ship first.

In [4]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = [f"{title}: {body}" for _, title, body in DOCS]
doc_emb = embedder.encode(texts, normalize_embeddings=True)

def vector_search(query):
    """Rank all documents by cosine similarity to the query (pure vector search).

    Embeds the query with the same model used for the documents, then scores
    every document by the dot product of the two normalized embeddings
    (which equals cosine similarity). Returns the ranking: positions into
    DOCS, best first. The top hit is the document whose meaning is closest
    to the query, not necessarily the one with matching words.
    """
    q = embedder.encode([query], normalize_embeddings=True)[0]
    return list(np.argsort(-(doc_emb @ q)))

q1, gold1 = QUERIES[0]
r = vector_search(q1)
show(q1, r)
print("Gold doc (returns-intl) rank:", gold_rank(r, gold1))

Query: What is the refund window for international orders?
  1. Domestic Returns (id=returns-domestic)
  2. International Returns (id=returns-intl)
  3. Return Shipping Costs (id=return-shipping-cost)
Gold doc (returns-intl) rank: 2


The top hit is **Domestic Returns** — the 30-day policy. A bot built on this retrieval answers **30 days**, confidently wrong. The correct document is ranked #2, close but not first, and "close" does not count in top-1 generation.

How bad is it across all four queries?

In [5]:
for q, gold in QUERIES:
    r = vector_search(q)
    print(f"{gold:15s} gold rank={gold_rank(r, gold)}  "
          f"top3={[DOCS[i][0] for i in r[:3]]}")

returns-intl    gold rank=2  top3=['returns-domestic', 'returns-intl', 'return-shipping-cost']
faulty-returns  gold rank=1  top3=['faulty-returns', 'warranty', 'returns-domestic']
lost-parcel     gold rank=4  top3=['faulty-returns', 'returns-domestic', 'returns-intl']
exchanges       gold rank=1  top3=['exchanges', 'faulty-returns', 'return-shipping-cost']


Vector search gets 3 out of 4 gold documents into the top 3 — but the hero query fails at rank 1, and the lost-parcel query misses the top 3 entirely (rank 4). That is the bug from the article: the document exists, retrieval still fails.

## Step 2: BM25 alone — keyword search

**BM25** is classic keyword ranking: it scores a document by how often your query terms appear in it, discounted by how common each term is across the corpus, and normalized for document length. Great at exact terms and codes. Blind to synonyms and meaning.

In [6]:
from rank_bm25 import BM25Okapi

def tokenize(t):
    """Split text into lowercase word tokens for BM25.

    Lowercases everything and keeps only letters and digits, so 'Refund,'
    and 'refund' count as the same word. BM25 works on these tokens, not
    on meaning: it can only match words that literally appear.
    """
    return re.findall(r"[a-z0-9]+", t.lower())

bm25 = BM25Okapi([tokenize(f"{t} {x}") for _, t, x in DOCS])

def bm25_search(query):
    """Rank all documents with BM25 keyword search.

    Scores each document by how often the query's words appear in it,
    discounted by how common each word is across the corpus and normalized
    for document length. Returns the ranking, best first. Strong on exact
    terms and codes; blind to synonyms: 'refund' never matches
    'restitution' here.
    """
    scores = bm25.get_scores(tokenize(query))
    return list(np.argsort(-np.array(scores)))

for q, gold in QUERIES:
    r = bm25_search(q)
    print(f"{gold:15s} gold rank={gold_rank(r, gold)}  "
          f"top3={[DOCS[i][0] for i in r[:3]]}")

returns-intl    gold rank=3  top3=['returns-domestic', 'refund-method', 'returns-intl']
faulty-returns  gold rank=4  top3=['returns-domestic', 'returns-intl', 'return-shipping-cost']
lost-parcel     gold rank=12  top3=['damaged-box', 'store-credit', 'returns-intl']
exchanges       gold rank=5  top3=['return-shipping-cost', 'lost-parcel', 'size-guide']


BM25 alone is much worse: only 1 of 4 gold documents in the top 3, and the lost-parcel document sinks to rank 12. The query's words (*money back, never arrives*) simply do not appear in the gold document (*remitted in full, undelivered*). Keyword search cannot cross a vocabulary gap.

## Step 3: Hybrid search — BM25 + vector fused with RRF

**RRF (Reciprocal Rank Fusion)** combines ranked lists without normalizing scores: each document gets `1 / (60 + rank)` from each list, and the scores are added. It is the standard fusion method because it is simple and robust.

Expectation: hybrid should beat both parents. Watch what actually happens.

In [7]:
def rrf(rank_lists, k=60):
    """Fuse several rankings into one with Reciprocal Rank Fusion.

    rank_lists: a list of rankings, e.g. [vector ranking, BM25 ranking].
    Each document scores 1 / (k + rank) from each list, and the scores are
    added up. A document ranked #1 in both lists beats one ranked #1 in a
    single list. k=60 is the standard constant: it softens the difference
    between nearby ranks. No score normalization is needed, which is why
    RRF is the default way to combine search methods.
    """
    scores = {}
    for ranks in rank_lists:
        for rank, idx in enumerate(ranks):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

for q, gold in QUERIES:
    r = rrf([vector_search(q), bm25_search(q)])
    print(f"{gold:15s} gold rank={gold_rank(r, gold)}  "
          f"top3={[DOCS[i][0] for i in r[:3]]}")

returns-intl    gold rank=2  top3=['returns-domestic', 'returns-intl', 'refund-method']
faulty-returns  gold rank=2  top3=['returns-domestic', 'faulty-returns', 'returns-intl']
lost-parcel     gold rank=6  top3=['returns-domestic', 'returns-intl', 'damaged-box']
exchanges       gold rank=2  top3=['return-shipping-cost', 'exchanges', 'price-match']


Hybrid does **not** rescue it. Mean gold rank goes from 2.0 (vector alone) to 3.0 — fusing with a weak signal drags the good signal down. The lost-parcel document falls from rank 4 to rank 6. Hybrid search is not a silver bullet; it only helps when both signals are at least decent. Lesson: measure the fusion, do not assume it.

## Step 4: Synonym expansion — a dictionary, no model needed

Add known equivalents to the query before searching: *refund -> reimbursement, restitution; international -> cross-border, overseas, abroad; never arrives -> undelivered, lost in transit*. Transparent, cheap, and someone has to maintain the dictionary.

In [8]:
SYNONYMS = {
 "refund": ["refund", "reimbursement", "restitution", "repayment", "remitted"],
 "window": ["window", "period", "interval", "deadline"],
 "international": ["international", "cross-border", "overseas", "abroad"],
 "orders": ["orders", "shipments", "parcels", "consignments", "purchases"],
 "send back": ["send back", "return", "remit"],
 "faulty": ["faulty", "defective", "damaged", "non-conforming", "defect"],
 "money back": ["money back", "refund", "reimbursement", "remitted"],
 "never arrives": ["never arrives", "fails to arrive", "undelivered",
                   "lost in transit"],
 "charge": ["charge", "cost", "fee", "carriage"],
 "returning": ["returning", "return", "remit"],
 "shoes": ["shoes", "footwear"],
 "don't fit": ["don't fit", "do not fit", "wrong size", "unsuitable size"],
 "how long": ["how long", "period", "interval", "days"],
 "parcel": ["parcel", "consignment", "shipment"],
}

def expand_query(q):
    """Append known synonyms to the query before searching.

    Scans the query for phrases in the SYNONYMS dictionary and adds their
    equivalents, e.g. 'refund' also adds 'reimbursement' and 'restitution'.
    Returns the longer query string. Transparent and cheap, but it can
    lift a distractor document just as much as the gold one.
    """
    ql = q.lower()
    extra = []
    for phrase, syns in SYNONYMS.items():
        if phrase in ql:
            extra.extend(syns)
    return q + " " + " ".join(extra) if extra else q

In [9]:
print("Expanded Q1:", expand_query(QUERIES[0][0]), "\n")
for q, gold in QUERIES:
    r = vector_search(expand_query(q))
    print(f"{gold:15s} gold rank={gold_rank(r, gold)}  "
          f"top3={[DOCS[i][0] for i in r[:3]]}")

Expanded Q1: What is the refund window for international orders? refund reimbursement restitution repayment remitted window period interval deadline international cross-border overseas abroad orders shipments parcels consignments purchases 
returns-intl    gold rank=2  top3=['returns-domestic', 'returns-intl', 'refund-method']
faulty-returns  gold rank=1  top3=['faulty-returns', 'warranty', 'returns-domestic']
lost-parcel     gold rank=1  top3=['lost-parcel', 'faulty-returns', 'returns-intl']


exchanges       gold rank=1  top3=['exchanges', 'size-guide', 'faulty-returns']


Expansion fixes the lost-parcel query completely (rank 4 -> 1) and gets all four gold documents into the top 3. But look at Q1: the domestic document is **still #1**. Expansion lifted the distractor too — it shares just as many expanded terms. Expansion fixes synonym gaps; it cannot fix a query whose distractor matches even better.

## Step 5: LLM query rewriting — restate the query in document language

Same idea as expansion, but the LLM generates the vocabulary instead of a dictionary. Two few-shot examples teach the mapping pattern; the model does the rest. This runs locally on Qwen2.5-0.5B-Instruct — no API key. We apply it only to the two queries that are still failing, because that is the point: diagnose first, then fix where the failure is.

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
llm = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float32, low_cpu_mem_usage=True)
llm.eval()

def generate(system, user, max_new=60):
    """Get one greedy (deterministic) completion from the local Qwen model.

    system: the instruction that sets the model's job (rewriter, writer...).
    user: the actual task, e.g. the question to rewrite.
    max_new: maximum new tokens to generate.
    Builds a chat prompt from the two messages, generates with greedy
    decoding (do_sample=False, so the same input always gives the same
    output), and returns the reply text only.
    """
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt")
    out = llm.generate(**ids, max_new_tokens=max_new, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:],
                      skip_special_tokens=True).strip()

In [11]:
REWRITE_SYS = ("You rewrite customer questions into the formal wording a "
               "store policy document would use. Reply with only the "
               "rewritten query, nothing else.")
REWRITE_USER = (
    "Example 1: Question: How do I get my money back for a late delivery? "
    "Rewritten: reimbursement for delayed shipment\n"
    "Example 2: Question: What is the return window for overseas orders? "
    "Rewritten: restitution interval for consignments dispatched abroad\n"
    "Question: {q} Rewritten:")

rewrite_ranks = {}
for q, gold in [QUERIES[0], QUERIES[2]]:
    rw = generate(REWRITE_SYS, REWRITE_USER.format(q=q), max_new=40)
    print("Original: ", q)
    print("Rewritten:", rw)
    r = vector_search(rw)
    rewrite_ranks[gold] = gold_rank(r, gold)
    print(f"Gold rank: {rewrite_ranks[gold]}  "
          f"top3: {[DOCS[i][0] for i in r[:3]]}\n")

Original:  What is the refund window for international orders?
Rewritten: refund period for international shipments


Gold rank: 1  top3: ['returns-intl', 'returns-domestic', 'vat-customs']


Original:  Can I get my money back if the parcel never arrives?
Rewritten: reimbursement for non-delivery


Gold rank: 4  top3: ['return-shipping-cost', 'faulty-returns', 'payment-methods']


Rewriting fixes Q1 — the gold document finally takes rank 1, because the rewritten query speaks the document's language (*refund period for international shipments*) instead of the customer's. It does not fix Q3 (*reimbursement for non-delivery* is too generic). Again: different failures, different fixes.

## Step 6: HyDE — and its failure mode

**HyDE (Hypothetical Document Embeddings):** ask the LLM to write a hypothetical document that would answer the question, then embed *that* instead of the short query. The fake document uses document-like language, so it should land close to the real one.

There is a catch, and this notebook shows it on purpose. Watch the generated text.

In [12]:
HYDE_SYS = ("You write short passages from a store policy document. Write "
            "about the topic in formal policy language. Do NOT state any "
            "specific number of days, since you do not know it. Reply with "
            "only the passage, nothing else.")

hyde_ranks = {}
for q, gold in [QUERIES[0], QUERIES[2]]:
    hd = generate(HYDE_SYS, "Write a short policy paragraph about this topic: " + q,
                  max_new=90)
    print("Hypothetical doc:", hd, "\n")
    r = vector_search(hd)
    hyde_ranks[gold] = gold_rank(r, gold)
    print(f"Gold rank: {hyde_ranks[gold]}  "
          f"top3: {[DOCS[i][0] for i in r[:3]]}\n")

Hypothetical doc: The refund window for international orders typically begins on the day following receipt of your order and ends within 30 days after shipment. This period allows customers to review their purchase and make any necessary adjustments before finalizing their payment and receiving the goods. 
Gold rank: 2  top3: ['returns-domestic', 'returns-intl', 'refund-method']


Hypothetical doc: The policy prohibits the return of any money received for goods that have not been delivered to the customer's address within the specified time frame. This includes parcels that are lost or damaged during transit. If a customer receives a package that is not delivered as scheduled, they must contact the shipping company and provide proof of delivery to initiate the refund process. Failure to follow these procedures may result in legal consequences. 
Gold rank: 3  top3: ['faulty-returns', 'return-shipping-cost', 'lost-parcel']


HyDE helps Q3 a little (rank 4 -> 3) but does not fix Q1 (still rank 2). Worse, look at the Q1 hypothetical document: it states **"30 days"** — the wrong fact — even though the prompt told it not to state any number. The small model guessed the common case, and that guess is now baked into the search. HyDE's hypothetical document reflects the model's parametric knowledge, which can be wrong. When the model guesses wrong, HyDE retrieves *toward* the wrong answer.

## Scoreboard

Gold-document rank per method (lower is better; rank 1 = the bot would answer from the right document).

In [13]:
methods = {}
for q, gold in QUERIES:
    methods[gold] = {
        "vector": gold_rank(vector_search(q), gold),
        "bm25": gold_rank(bm25_search(q), gold),
        "hybrid-rrf": gold_rank(rrf([vector_search(q), bm25_search(q)]), gold),
        "expanded": gold_rank(vector_search(expand_query(q)), gold),
        "rewrite": rewrite_ranks.get(gold, "-"),
        "hyde": hyde_ranks.get(gold, "-"),
    }

names = ["vector", "bm25", "hybrid-rrf", "expanded", "rewrite", "hyde"]
print(f"{'query':15s}" + "".join(f"{n:>11s}" for n in names))
for gold, row in methods.items():
    print(f"{gold:15s}" + "".join(
        f"{str(row[n]):>11s}" for n in names))
print()
for n in names:
    ranks = [row[n] for row in methods.values() if row[n] != "-"]
    print(f"{n:11s} mean rank={sum(ranks)/len(ranks):.2f}  "
          f"recall@3={sum(r <= 3 for r in ranks)}/{len(ranks)}")

query               vector       bm25 hybrid-rrf   expanded    rewrite       hyde
returns-intl             2          3          2          2          1          2
faulty-returns           1          4          2          1          -          -
lost-parcel              4         12          6          1          4          3
exchanges                1          5          2          1          -          -
vector      mean rank=2.00  recall@3=3/4
bm25        mean rank=6.00  recall@3=1/4
hybrid-rrf  mean rank=3.00  recall@3=3/4
expanded    mean rank=1.25  recall@3=4/4
rewrite     mean rank=2.50  recall@3=1/2
hyde        mean rank=2.50  recall@3=2/2


## What this demo shows

- **Vector search alone fails the hero query.** The domestic 30-day policy outranks the international 60-day policy, so the bot answers 30 days. The document existed the whole time.
- **BM25 alone is worse here** — the vocabulary gap is exactly what keyword search cannot cross.
- **Hybrid RRF does not rescue it.** Fusing with a weak signal drags the good one down (mean rank 2.0 -> 3.0).
- **Synonym expansion fixes the synonym gap** (lost-parcel 4 -> 1, recall@3 4/4) but cannot dethrone a distractor that matches just as well.
- **LLM rewriting fixes the phrasing gap** (Q1 2 -> 1) — the one method that gets the gold document to rank 1 on the hero query.
- **HyDE can backfire.** Its hypothetical document hallucinated "30 days", the wrong fact, and pushed retrieval toward the wrong answer.

No single fix wins everywhere. That is the point of the debugging framework in the companion article: find which stage failed, name the failure type, and apply the fix that matches it — then measure recall@k before and after.

## Share it

- **In Colab:** File -> Download -> `.ipynb` to keep the notebook. For a static HTML copy you can share with anyone, run `jupyter nbconvert --to html notebook.ipynb` from a terminal (that is how the HTML version delivered with this notebook was made).
- **Try it yourself:** add your own documents to `DOCS`, your own question to `QUERIES`, and watch which method fixes it — or fails to.